In [1]:
import sys
import os
import nest_asyncio

project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

nest_asyncio.apply()

print(f'Project root set to: {project_root}')

Project root set to: /Users/rob./Downloads/Code/Carton Caps Chatbot


In [2]:
from app.db.connection import SessionLocal, engine, Base
from app.db.repository import Repository

Base.metadata.create_all(bind=engine)

db = SessionLocal()
try:
    conv = Repository.create_conversation(db, user_id='test_user_notebook', entry_point='notebook_test')
    conv_id = str(conv.id)
    print(f'Created Conversation ID: {conv_id}')
    msg = Repository.save_message(db, conv_id, role='user', content='Hello from notebook!')
    print(f'Saved Message ID: {str(msg.id)}')
finally:
    db.close()

Created Conversation ID: c_44d3fd3c
Saved Message ID: m_96657bd9


In [3]:
from app.db.connection import SessionLocal
from app.guardrails.input_guardrail import InputGuardrail
from app.services.recommendation_service import RecommendationService
from app.retrieval.hybrid_retriever import HybridRetriever
from app.services.context_service import ContextService
from app.services.assistant_service import AssistantService

db = SessionLocal()
try:
    user_input = 'What snacks support my school?'
    clean_input = InputGuardrail.validate(user_input)
    print(f'[1] Guardrail Output: {clean_input}')
    intent = RecommendationService.classify_intent(clean_input)
    print(f'[2] Classified Intent: {intent}')
    facts = HybridRetriever.retrieve(db, intent, clean_input)
    print(f'[3] SQL Retrieved Facts: {facts}')
    context = ContextService.prepare_context(intent, facts)
    print(f'[4] Prepared Prompt Context:\n{context}')
    msg_db, intent, reply, facts = AssistantService.process_message(db, conv_id, clean_input)
    print(f'[5] Final Assistant Reply:\n{reply}')
finally:
    db.close()

[1] Guardrail Output: What snacks support my school?
[2] Classified Intent: PRODUCT_QUERY
[3] SQL Retrieved Facts: [{'id': 1, 'name': 'Frosted Flakes Cereal', 'price': 3.79}, {'id': 2, 'name': 'Granola Cereal Bars', 'price': 5.92}, {'id': 3, 'name': 'Macaroni & Cheese Box', 'price': 9.61}, {'id': 4, 'name': 'Instant Oatmeal Packets', 'price': 5.26}, {'id': 5, 'name': 'Fruit Snacks Pouch', 'price': 7.44}]
[4] Prepared Prompt Context:
You are Capper, a helpful assistant for the Carton Caps app.
Intent: PRODUCT_QUERY
Database Facts:
- Product: Frosted Flakes Cereal, Price: $3.79
- Product: Granola Cereal Bars, Price: $5.92
- Product: Macaroni & Cheese Box, Price: $9.61
- Product: Instant Oatmeal Packets, Price: $5.26
- Product: Fruit Snacks Pouch, Price: $7.44

[5] Final Assistant Reply:
Here are some snacks that you can consider for supporting your school:

1. **Granola Cereal Bars** - Priced at $5.92, these can be a great snack option for students.
2. **Fruit Snacks Pouch** - Available 

In [4]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

res = client.get('/')
print('Health Check:', res.json())

create_res = client.post('/v1/chat/conversations', json={'entry_point': 'notebook_test'})
conv_data = create_res.json()
print('Create Conversation Response:', conv_data)

conv_id = conv_data['conversation_id']

msg_res = client.post(
    f'/v1/chat/conversations/{conv_id}/messages',
    json={'content': "Can I buy Annie's Macaroni?"}
)
print('Send Message Response:', msg_res.json())

Health Check: {'status': 'online', 'service': 'Carton Caps AI Chat Agent'}
Create Conversation Response: {'conversation_id': 'c_7976b917', 'greeting': 'Hi! I am Capper. I can help you find products that support your school!'}
Send Message Response: {'message_id': 'm_fbb94c11', 'role': 'assistant', 'intent': 'PRODUCT_QUERY', 'reply': "I don't have information on Annie's Macaroni specifically, but I do have a Macaroni & Cheese Box available for $9.61. Would you like to know more about it?", 'retrieved_data': [{'id': 1, 'name': 'Frosted Flakes Cereal', 'price': 3.79}, {'id': 2, 'name': 'Granola Cereal Bars', 'price': 5.92}, {'id': 3, 'name': 'Macaroni & Cheese Box', 'price': 9.61}, {'id': 4, 'name': 'Instant Oatmeal Packets', 'price': 5.26}, {'id': 5, 'name': 'Fruit Snacks Pouch', 'price': 7.44}]}


In [5]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# Start a new conversation
create_res = client.post('/v1/chat/conversations', json={'entry_point': 'interactive'})
conv_id = create_res.json()['conversation_id']
print(f'Conversation started: {conv_id}')
print('Type your message below and run this cell. Change the prompt and re-run for multi-turn.\n')

# --- Change this prompt and re-run the cell ---
user_prompt = "How do I refer a friend?"



# ----------------------------------------------

res = client.post(
    f'/v1/chat/conversations/{conv_id}/messages',
    json={'content': user_prompt}
)
data = res.json()
print(f'You: {user_prompt}')
print(f'Capper [{data["intent"]}]: {data["reply"]}')

Conversation started: c_e6ea9efa
Type your message below and run this cell. Change the prompt and re-run for multi-turn.

You: How do I refer a friend?
Capper [FAQ_QUERY]: You can refer a friend directly from the Carton Caps app by following these steps:

1. Tap on the account icon to see account options.
2. Tap "Invite Friends" from the menu.
3. Copy the referral code and send it to your friend, or share a link using the buttons in the "Share Now" section.
4. Your friend must install the app using your link or sign up using your code.

Once they do, you'll both be eligible for a bonus!
